# Sally Prompt Analysis

Fetch recent extraction logs from the Sally admin API and analyze prompt/response quality.

**Requires:** `pip install requests pandas`

**Auth:** Set `SALLY_API_TOKEN` in your environment. Create a token from Admin / API Tokens
and use it as `Authorization: Bearer <token>` for the admin API requests.

In [1]:
import requests
import pandas as pd
import json
import os
from IPython.display import display, HTML

# Switch between local and dev:
BASE_URL = "http://localhost:8080"           # local
# BASE_URL = "https://dev.spexxtool.com"    # dev server

api_token = os.environ.get("SALLY_API_TOKEN", "").strip()
if not api_token:
    raise RuntimeError("SALLY_API_TOKEN is not set. Create a token in Admin / API Tokens, then export SALLY_API_TOKEN=<token> before starting Jupyter.")

session = requests.Session()
session.headers.update({"Authorization": f"Bearer {api_token}"})

def read_json_response(resp):
    resp.raise_for_status()
    content_type = resp.headers.get("content-type", "")
    if "application/json" not in content_type.lower():
        preview = resp.text.strip().replace("\n", " ")[:300]
        raise RuntimeError(
            f"Expected JSON from {resp.url}, got {resp.status_code} {content_type!r}. "
            f"This usually means the request hit login/forbidden HTML. "
            f"Check BASE_URL and SALLY_API_TOKEN. Preview: {preview!r}"
        )
    return resp.json()

print(f"Targeting: {BASE_URL}")

Targeting: http://localhost:8080


## Load recent logs

In [2]:
resp = session.get(f"{BASE_URL}/admin/api/extraction-logs", params={"limit": 200})
data = read_json_response(resp)

df = pd.DataFrame(data["logs"])
if df.empty:
    print("No logs found.")
else:
    # Convert types
    df["CreatedAt"] = pd.to_datetime(df["CreatedAt"])
    df["DurationMS"] = df["DurationMS"].astype(int)
    print(f"Loaded {len(df)} logs from {df['CreatedAt'].min().date()} to {df['CreatedAt'].max().date()}")
    display(df[["CreatedAt","PromptVersion","Provider","Model","Success","DurationMS",
                "PromptTokens","CompletionTokens","MissingFieldCount","PageURL"]].head(20))

Loaded 47 logs from 2026-05-06 to 2026-05-11


,CreatedAt,PromptVersion,Provider,Model,Success,DurationMS,PromptTokens,CompletionTokens,MissingFieldCount,PageURL
0,2026-05-11 02:19:05+00:00,extract-spec-v2,ollama,qwen2.5:7b,True,9888,0,0,0,https://www.lowes.com/pd/MRCOOL-DIY-5TH-Gen-9-...
1,2026-05-11 02:15:28+00:00,extract-spec-v2,ollama,qwen2.5:7b,True,9174,0,0,0,https://www.lowes.com/pd/BHI-Single-Zone-12000...
2,2026-05-11 02:11:42+00:00,extract-spec-v2,ollama,qwen2.5:7b,True,9028,0,0,2,https://www.lowes.com/pd/JELD-WEN-V-4500-Singl...
3,2026-05-11 02:10:26+00:00,extract-spec-v2,ollama,qwen2.5:7b,True,7587,0,0,0,https://example.com/product
4,2026-05-11 02:07:42+00:00,extract-spec-v2,anthropic,claude-haiku-4-5,True,2820,2016,470,0,https://example.com/product
5,2026-05-11 02:07:17+00:00,extract-spec-v2,anthropic,claude-haiku-4-5,True,3030,2016,467,0,https://example.com/product
6,2026-05-10 22:37:42+00:00,extract-spec-v2,anthropic,claude-haiku-4-5,True,2724,2017,480,0,https://example.com/product
7,2026-05-10 22:29:33+00:00,extract-spec-v2,anthropic,claude-haiku-4-5,True,6884,9232,675,1,https://www.lowes.com/pd/JELD-WEN-V-4500-Singl...
8,2026-05-10 22:23:57+00:00,extract-spec-v2,anthropic,claude-haiku-4-5,True,7702,9219,662,0,https://www.lowes.com/pd/JELD-WEN-V-4500-Singl...
9,2026-05-10 21:48:43+00:00,extract-spec-v2,anthropic,claude-haiku-4-5,True,6763,9219,691,0,https://www.lowes.com/pd/JELD-WEN-V-4500-Singl...


## Summary stats

In [ ]:
total = len(df)
successes = df["Success"].sum()
print(f"Success rate: {successes}/{total} = {successes/total*100:.1f}%")
print(f"Avg duration (success): {df[df['Success']]['DurationMS'].mean():.0f} ms")
print(f"Avg prompt tokens:      {df['PromptTokens'].mean():.0f}")
print(f"Avg completion tokens:  {df['CompletionTokens'].mean():.0f}")
print(f"Avg missing fields:     {df['MissingFieldCount'].mean():.2f}")

In [ ]:
# Break down by prompt version — this is where A/B comparison lives
by_version = df.groupby("PromptVersion").agg(
    count=("Success", "count"),
    success_rate=("Success", "mean"),
    avg_dur_ms=("DurationMS", "mean"),
    avg_missing=("MissingFieldCount", "mean"),
    avg_prompt_tok=("PromptTokens", "mean"),
    avg_completion_tok=("CompletionTokens", "mean"),
).round(2)
by_version["success_rate"] = (by_version["success_rate"] * 100).round(1).astype(str) + "%"
display(by_version)

## Errors

In [ ]:
errors_df = df[~df["Success"]].copy()
print(f"{len(errors_df)} errors")
if not errors_df.empty:
    display(errors_df[["CreatedAt","PromptVersion","Provider","Error","PageURL"]].head(20))

## Inspect a single extraction (prompt + response)

Set `REQUEST_ID` to the `RequestID` of any row above.

In [ ]:
# Paste a RequestID from above
REQUEST_ID = df.iloc[18]["RequestID"] if not df.empty else ""

resp = session.get(f"{BASE_URL}/admin/api/extraction-logs/{REQUEST_ID}")
log = read_json_response(resp)

print(f"Request:  {log['RequestID']}")
print(f"Version:  {log['PromptVersion']}")
print(f"Provider: {log['Provider']} / {log['Model']}")
print(f"Success:  {log['Success']}")
print(f"URL:      {log['PageURL']}")
if not log.get("Success"):
    print(f"Error:    {log['Error']}")

In [ ]:
# Show the prompt sent to the LLM
prompt_raw = log.get("PromptText", "")
try:
    prompt_parsed = json.loads(prompt_raw)
    print(json.dumps(prompt_parsed, indent=2))
except json.JSONDecodeError:
    print(prompt_raw)

In [ ]:
# Show the raw LLM response
resp_raw = log.get("ResponseText", "")
try:
    resp_parsed = json.loads(resp_raw)
    print(json.dumps(resp_parsed, indent=2))
except json.JSONDecodeError:
    print(resp_raw)

## A/B comparison

Once you have two prompt versions in the data, compare them side-by-side.
Set `VERSION_A` and `VERSION_B` to the `PromptVersion` strings you want to compare.

In [3]:
VERSION_A = "extract-spec-v1"
VERSION_B = "extract-spec-v2"   # change when a new version exists

a_df = df[df["PromptVersion"] == VERSION_A]
b_df = df[df["PromptVersion"] == VERSION_B]

def stats(d, label):
    if d.empty:
        return {"version": label, "n": 0}
    return {
        "version": label,
        "n": len(d),
        "success_rate": f"{d['Success'].mean()*100:.1f}%",
        "avg_dur_ms": f"{d['DurationMS'].mean():.0f}",
        "avg_missing": f"{d['MissingFieldCount'].mean():.2f}",
        "avg_prompt_tok": f"{d['PromptTokens'].mean():.0f}",
        "avg_completion_tok": f"{d['CompletionTokens'].mean():.0f}",
    }

display(pd.DataFrame([stats(a_df, VERSION_A), stats(b_df, VERSION_B)]).set_index("version"))

,n,success_rate,avg_dur_ms,avg_missing,avg_prompt_tok,avg_completion_tok
version,,,,,,
extract-spec-v1,10,60.0%,6439,0.20,4395,464
extract-spec-v2,34,100.0%,6723,0.09,7291,567
